In [52]:
import yfinance as yf
import pandas as pd
import time

def process_nifty_segment(tickers, filename):
    dashboard_data = []
    daily_status_data = []
    
    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        workbook = writer.book
        # Formatting colors
        yellow = workbook.add_format({'bg_color': '#FFFF00'}) # Buy
        green  = workbook.add_format({'bg_color': '#C6EFCE'}) # Profit
        red    = workbook.add_format({'bg_color': '#FFC7CE'}) # Exit/Loss

        stock_dfs = {}

        for s in tickers:
            try:
                print(f"Processing: {s}...")
                tk = yf.Ticker(s)
                df = tk.history(period="10y", interval="1d")
                if df is None or df.empty: continue
                if df.index.tz is not None: df.index = df.index.tz_localize(None)

                df['50 DMA'] = df['Close'].rolling(window=50).mean()
                df['100 DMA'] = df['Close'].rolling(window=100).mean()
                df['200 DMA'] = df['Close'].rolling(window=200).mean()

                status, targets = [], []
                in_trade, target_p = False, 0
                trade_count, profit_count, exit_count = 0, 0, 0
                
                # Logic to track prices for the Daily Check
                buy_signal_price = None

                for i in range(len(df)):
                    p = float(df['Close'].iloc[i])
                    m50, m100, m200 = float(df['50 DMA'].iloc[i]), float(df['100 DMA'].iloc[i]), float(df['200 DMA'].iloc[i])
                    
                    if pd.isna(m200):
                        status.append("WAIT"); targets.append(None); continue

                    is_bull = (p > m50 > m100 > m200) and (p < 1.1 * m200)
                    is_bear = (p < m50 < m100 < m200)

                    if not in_trade:
                        if is_bull:
                            in_trade, target_p, trade_count = True, p * 1.0628, trade_count + 1
                            buy_signal_price = p # Capture price when signal hits
                            status.append("BUY"); targets.append(round(target_p, 2))
                        else:
                            status.append("WAIT"); targets.append(None)
                    else:
                        if p >= target_p:
                            status.append("PROFIT"); targets.append(round(target_p, 2))
                            in_trade, profit_count = False, profit_count + 1
                        elif is_bear:
                            status.append("EXIT"); targets.append(None)
                            in_trade, exit_count = False, exit_count + 1
                        else:
                            status.append("HOLDING"); targets.append(round(target_p, 2))

                df['Strategy Status'] = status
                df['Target Value'] = targets
                
                # Update Dashboard
                dashboard_data.append([s.replace(".NS",""), trade_count, profit_count, exit_count])
                
                # Update Daily Check with Prices
                current_price = float(df['Close'].iloc[-1])
                daily_status_data.append([
                    s.replace(".NS",""), 
                    df['Strategy Status'].iloc[-2], 
                    df['Strategy Status'].iloc[-1],
                    round(buy_signal_price, 2) if in_trade else "N/A", # Price at entry
                    round(current_price, 2) # Current Market Price
                ])

                # Sort Descending for Stock Tabs
                df_desc = df.sort_index(ascending=False).reset_index()
                stock_dfs[s.replace(".NS", "")[:31]] = df_desc
                time.sleep(0.05)
                
            except Exception as e:
                print(f"Error on {s}: {e}")

        # 1. WRITE DASHBOARD
        df_dash = pd.DataFrame(dashboard_data, columns=['Stock', 'Total Trades', 'Profitable', 'Loss/Exit'])
        df_dash.to_excel(writer, sheet_name='DASHBOARD', index=False)
        
        # 2. WRITE DAILY_CHECK (Updated Columns)
        df_daily = pd.DataFrame(daily_status_data, columns=['Stock', 'Yesterday Status', 'Today Status', 'Entry Price (Signal)', 'Current Price'])
        df_daily.to_excel(writer, sheet_name='DAILY_CHECK', index=False)

        # 3. WRITE STOCK TABS WITH COLORS
        for sheet_name, df_stock in stock_dfs.items():
            df_stock.to_excel(writer, sheet_name=sheet_name, index=False)
            ws = writer.sheets[sheet_name]
            
            # Formatting range (Row 1 to End, Column A to L)
            # Column K (Index 10) is "Strategy Status"
            last_row = len(df_stock)
            
            ws.conditional_format(1, 0, last_row, 11, {
                'type':     'formula',
                'criteria': '=$K2="BUY"', # Column K is fixed with $
                'format':   yellow
            })
            ws.conditional_format(1, 0, last_row, 11, {
                'type':     'formula',
                'criteria': '=$K2="PROFIT"',
                'format':   green
            })
            ws.conditional_format(1, 0, last_row, 11, {
                'type':     'formula',
                'criteria': '=$K2="EXIT"',
                'format':   red
            })

    print(f"Success! {filename} created.")

In [53]:
import yfinance as yf
import pandas as pd
import time

def process_nifty_car_final(tickers, filename):
    dashboard_data = []
    daily_status_data = []
    
    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        workbook = writer.book
        yellow = workbook.add_format({'bg_color': '#FFFF00', 'bold': True}) # BUY Signal
        green  = workbook.add_format({'bg_color': '#C6EFCE'}) # Recovery Zone
        red    = workbook.add_format({'bg_color': '#FFC7CE'}) # Falling Zone

        stock_dfs = {}

        for s in tickers:
            try:
                print(f"Analyzing {s}...")
                tk = yf.Ticker(s)
                df_full = tk.history(period="10y", interval="1d")
                if df_full.empty: continue
                if df_full.index.tz is not None: df_full.index = df_full.index.tz_localize(None)

                # 1. Find the Absolute All-Time High (ATH)
                ath_price = df_full['Close'].max()
                ath_date = df_full[df_full['Close'] == ath_price].index[0]
                df = df_full.loc[ath_date:].copy()

                # 2. CA (Cumulative Average) from ATH
                df['CA'] = df['Close'].expanding().mean()
                df['CA_Diff'] = df['CA'].diff()

                # 3. Calculate Consecutive Up-Days
                consec_up = []
                count = 0
                for diff in df['CA_Diff']:
                    if diff > 0:
                        count += 1
                    else:
                        count = 0 # Reset if CA falls
                    consec_up.append(count)
                df['Consec_Days'] = consec_up

                # 4. Trigger BUY Signal every 10 days
                # (Day 10, 20, 30...)
                status = []
                for val in df['Consec_Days']:
                    if val > 0 and val % 10 == 0:
                        status.append("BUY SIGNAL")
                    elif val > 0:
                        status.append("RECOVERY ZONE")
                    else:
                        status.append("FALLING ZONE")
                
                df['Strategy Status'] = status

                # 5. Prepare Dashboard & Daily Check
                last_row = df.iloc[-1]
                current_consec = int(last_row['Consec_Days'])
                days_remaining = 10 - (current_consec % 10) if last_row['Strategy Status'] != "FALLING ZONE" else "N/A"
                
                dashboard_data.append([s.replace(".NS",""), status.count("BUY SIGNAL")])
                daily_status_data.append([
                    s.replace(".NS",""), 
                    last_row['Strategy Status'],
                    current_consec,
                    days_remaining,
                    round(last_row['Close'], 2)
                ])

                # Prepare Excel Export
                df_export = df[['Close', 'CA', 'Consec_Days', 'Strategy Status']].sort_index(ascending=False).reset_index()
                stock_dfs[s.replace(".NS", "")[:31]] = df_export
                time.sleep(0.05)
                
            except Exception as e:
                print(f"Error on {s}: {e}")

        # Write Sheets
        df_dash = pd.DataFrame(dashboard_data, columns=['Stock', 'Total Buy Signals Hit'])
        df_dash.to_excel(writer, sheet_name='DASHBOARD', index=False)
        
        df_daily = pd.DataFrame(daily_status_data, columns=['Stock', 'Current Zone', 'Consecutive Days Up', 'Days to Next Signal', 'CMP'])
        df_daily.to_excel(writer, sheet_name='DAILY_CHECK', index=False)

        for sheet, data in stock_dfs.items():
            data.to_excel(writer, sheet_name=sheet, index=False)
            ws = writer.sheets[sheet]
            # Column E (Index 4) is Strategy Status
            ws.conditional_format(1, 4, len(data), 4, {'type': 'cell', 'criteria': '==', 'value': '"BUY SIGNAL"', 'format': yellow})
            ws.conditional_format(1, 4, len(data), 4, {'type': 'cell', 'criteria': '==', 'value': '"RECOVERY ZONE"', 'format': green})
            ws.conditional_format(1, 4, len(data), 4, {'type': 'cell', 'criteria': '==', 'value': '"FALLING ZONE"', 'format': red})

    print(f"Success! {filename} created.")

# 

In [54]:
nifty_50 = ["ADANIENT.NS", "ADANIPORTS.NS", "APOLLOHOSP.NS", "ASIANPAINT.NS", "AXISBANK.NS", "BAJAJ-AUTO.NS", "BAJFINANCE.NS", "BAJAJFINSV.NS", "BEL.NS", "BHARTIARTL.NS", "CIPLA.NS", "COALINDIA.NS", "DRREDDY.NS", "EICHERMOT.NS", "GRASIM.NS", "HCLTECH.NS", "HDFCBANK.NS", "HDFCLIFE.NS", "HINDALCO.NS", "HINDUNILVR.NS", "ICICIBANK.NS", "INDIGO.NS", "INFY.NS", "ITC.NS", "JIOFIN.NS", "JSWSTEEL.NS", "KOTAKBANK.NS", "LT.NS", "M&M.NS", "MARUTI.NS", "MAXHEALTH.NS", "NESTLEIND.NS", "NTPC.NS", "ONGC.NS", "POWERGRID.NS", "RELIANCE.NS", "SBILIFE.NS", "SHRIRAMFIN.NS", "SBIN.NS", "SUNPHARMA.NS", "TCS.NS", "TATACONSUM.NS", "TMPV.NS", "TATASTEEL.NS", "TECHM.NS", "TITAN.NS", "TRENT.NS", "ULTRACEMCO.NS", "WIPRO.NS"]
process_nifty_segment(nifty_50, "Nifty_50.xlsx")
process_nifty_car_final(nifty_50, "Nifty_50_CAR.xlsx")

Processing: ADANIENT.NS...
Processing: ADANIPORTS.NS...
Processing: APOLLOHOSP.NS...
Processing: ASIANPAINT.NS...
Processing: AXISBANK.NS...
Processing: BAJAJ-AUTO.NS...
Processing: BAJFINANCE.NS...
Processing: BAJAJFINSV.NS...
Processing: BEL.NS...
Processing: BHARTIARTL.NS...
Processing: CIPLA.NS...
Processing: COALINDIA.NS...
Processing: DRREDDY.NS...
Processing: EICHERMOT.NS...
Processing: GRASIM.NS...
Processing: HCLTECH.NS...
Processing: HDFCBANK.NS...
Processing: HDFCLIFE.NS...
Processing: HINDALCO.NS...
Processing: HINDUNILVR.NS...
Processing: ICICIBANK.NS...
Processing: INDIGO.NS...
Processing: INFY.NS...
Processing: ITC.NS...
Processing: JIOFIN.NS...
Processing: JSWSTEEL.NS...
Processing: KOTAKBANK.NS...
Processing: LT.NS...
Processing: M&M.NS...
Processing: MARUTI.NS...
Processing: MAXHEALTH.NS...
Processing: NESTLEIND.NS...
Processing: NTPC.NS...
Processing: ONGC.NS...
Processing: POWERGRID.NS...
Processing: RELIANCE.NS...
Processing: SBILIFE.NS...
Processing: SHRIRAMFIN.NS

In [55]:
next_50 = ["ABB.NS", "AMBUJACEM.NS", "BANKBARODA.NS", "BPCL.NS", "CANBK.NS", "DLF.NS", "GAIL.NS", "HAL.NS", "HAVELLS.NS", "HINDZINC.NS", "IOC.NS", "LICINDIA.NS", "LTIM.NS", "PIDILITIND.NS", "TATAPOWER.NS", "UNITDSPR.NS", "VEDL.NS", "ZOMATO.NS", "DMART.NS", "VBL.NS", "ADANIGREEN.NS", "ADANIPOWER.NS", "DIVISLAB.NS", "IRFC.NS", "CHOLAFIN.NS", "GODREJCP.NS", "ICICIGI.NS", "JINDALSTEL.NS", "LODHA.NS", "MANKIND.NS", "PAGEIND.NS", "PIIND.NS", "PNB.NS", "RECLTD.NS", "MOTHERSON.NS", "SIEMENS.NS", "SRF.NS", "TVSMOTOR.NS", "ZYDUSLIFE.NS", "UNIONBANK.NS", "BOSCHLTD.NS", "CGPOWER.NS", "COLPAL.NS", "INDHOTEL.NS", "JSWENERGY.NS", "LUPIN.NS", "MUTHOOTFIN.NS", "POLICYBZR.NS", "POLYCAB.NS"]
process_nifty_segment(next_50, "Nifty_Next50.xlsx")
process_nifty_car_final(next_50, "Nifty_Next50_CAR.xlsx")

Processing: ABB.NS...
Processing: AMBUJACEM.NS...
Processing: BANKBARODA.NS...
Processing: BPCL.NS...
Processing: CANBK.NS...
Processing: DLF.NS...
Processing: GAIL.NS...
Processing: HAL.NS...
Processing: HAVELLS.NS...
Processing: HINDZINC.NS...
Processing: IOC.NS...
Processing: LICINDIA.NS...


$LICINDIA.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Processing: LTIM.NS...
Processing: PIDILITIND.NS...
Processing: TATAPOWER.NS...
Processing: UNITDSPR.NS...
Processing: VEDL.NS...
Processing: ZOMATO.NS...


$ZOMATO.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Processing: DMART.NS...
Processing: VBL.NS...
Processing: ADANIGREEN.NS...
Processing: ADANIPOWER.NS...
Processing: DIVISLAB.NS...
Processing: IRFC.NS...
Processing: CHOLAFIN.NS...
Processing: GODREJCP.NS...
Processing: ICICIGI.NS...
Processing: JINDALSTEL.NS...
Processing: LODHA.NS...
Processing: MANKIND.NS...
Processing: PAGEIND.NS...
Processing: PIIND.NS...
Processing: PNB.NS...
Processing: RECLTD.NS...
Processing: MOTHERSON.NS...
Processing: SIEMENS.NS...
Processing: SRF.NS...
Processing: TVSMOTOR.NS...
Processing: ZYDUSLIFE.NS...
Processing: UNIONBANK.NS...
Processing: BOSCHLTD.NS...
Processing: CGPOWER.NS...
Processing: COLPAL.NS...
Processing: INDHOTEL.NS...
Processing: JSWENERGY.NS...
Processing: LUPIN.NS...
Processing: MUTHOOTFIN.NS...
Processing: POLICYBZR.NS...
Processing: POLYCAB.NS...
Success! Nifty_Next50.xlsx created.
Analyzing ABB.NS...
Analyzing AMBUJACEM.NS...
Analyzing BANKBARODA.NS...
Analyzing BPCL.NS...
Analyzing CANBK.NS...
Analyzing DLF.NS...
Analyzing GAIL.NS..

$LICINDIA.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Analyzing LTIM.NS...
Analyzing PIDILITIND.NS...
Analyzing TATAPOWER.NS...
Analyzing UNITDSPR.NS...
Analyzing VEDL.NS...
Analyzing ZOMATO.NS...


$ZOMATO.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Analyzing DMART.NS...
Analyzing VBL.NS...
Analyzing ADANIGREEN.NS...
Analyzing ADANIPOWER.NS...
Analyzing DIVISLAB.NS...
Analyzing IRFC.NS...
Analyzing CHOLAFIN.NS...
Analyzing GODREJCP.NS...
Analyzing ICICIGI.NS...
Analyzing JINDALSTEL.NS...
Analyzing LODHA.NS...
Analyzing MANKIND.NS...
Analyzing PAGEIND.NS...
Analyzing PIIND.NS...
Analyzing PNB.NS...
Analyzing RECLTD.NS...
Analyzing MOTHERSON.NS...
Analyzing SIEMENS.NS...
Analyzing SRF.NS...
Analyzing TVSMOTOR.NS...
Analyzing ZYDUSLIFE.NS...
Analyzing UNIONBANK.NS...
Analyzing BOSCHLTD.NS...
Analyzing CGPOWER.NS...
Analyzing COLPAL.NS...
Analyzing INDHOTEL.NS...
Analyzing JSWENERGY.NS...
Analyzing LUPIN.NS...
Analyzing MUTHOOTFIN.NS...
Analyzing POLICYBZR.NS...
Analyzing POLYCAB.NS...
Success! Nifty_Next50_CAR.xlsx created.


In [56]:
midcap_pt1 = ["MUTHOOTFIN.NS", "UNIONBANK.NS", "INDUSTOWER.NS", "BSE.NS", "IDEA.NS", "CUMMINSIND.NS", "HDFCAMC.NS", "ASHOKLEY.NS", "IDBI.NS", "INDIANB.NS", "POLYCAB.NS", "HEROMOTOCO.NS", "LUPIN.NS", "POWERINDIA.NS", "GMRAIRPORT.NS", "MARICO.NS", "HINDPETRO.NS", "ICICIPRULI.NS", "SWIGGY.NS", "GVT&D.NS", "DABUR.NS", "WAAREEENER.NS", "BHEL.NS", "ABCAPITAL.NS", "PERSISTENT.NS", "MANKIND.NS", "SRF.NS", "BHARTIHEXA.NS", "BHARATFORG.NS", "POLICYBZR.NS", "NYKAA.NS", "NHPC.NS", "AUBANK.NS", "OIL.NS", "TORNTPOWER.NS", "BANKINDIA.NS", "SBICARD.NS", "INDUSINDBK.NS", "PAYTM.NS", "UNOMINDA.NS", "LTF.NS", "FEDERALBNK.NS", "IDFCFIRSTB.NS", "NMDC.NS", "DIXON.NS", "FORTIS.NS", "GICRE.NS", "COROMANDEL.NS", "IOB.NS", "AUROPHARMA.NS"]
process_nifty_segment(midcap_pt1, "Midcap150_Part1.xlsx")
process_nifty_car_final(midcap_pt1, "Midcap150_Part1_CAR.xlsx")

Processing: MUTHOOTFIN.NS...
Processing: UNIONBANK.NS...
Processing: INDUSTOWER.NS...
Processing: BSE.NS...
Processing: IDEA.NS...
Processing: CUMMINSIND.NS...
Processing: HDFCAMC.NS...
Processing: ASHOKLEY.NS...
Processing: IDBI.NS...
Processing: INDIANB.NS...
Processing: POLYCAB.NS...
Processing: HEROMOTOCO.NS...
Processing: LUPIN.NS...
Processing: POWERINDIA.NS...
Processing: GMRAIRPORT.NS...
Processing: MARICO.NS...
Processing: HINDPETRO.NS...
Processing: ICICIPRULI.NS...
Processing: SWIGGY.NS...
Processing: GVT&D.NS...
Processing: DABUR.NS...
Processing: WAAREEENER.NS...
Processing: BHEL.NS...
Processing: ABCAPITAL.NS...
Processing: PERSISTENT.NS...
Processing: MANKIND.NS...
Processing: SRF.NS...
Processing: BHARTIHEXA.NS...
Processing: BHARATFORG.NS...
Processing: POLICYBZR.NS...
Processing: NYKAA.NS...
Processing: NHPC.NS...
Processing: AUBANK.NS...
Processing: OIL.NS...
Processing: TORNTPOWER.NS...
Processing: BANKINDIA.NS...
Processing: SBICARD.NS...
Processing: INDUSINDBK.NS.

In [57]:
midcap_pt2 = ["SAIL.NS", "YESBANK.NS", "PRESTIGE.NS", "LLOYDSME.NS", "ALKEM.NS", "RVNL.NS", "NATIONALUM.NS", "MRF.NS", "MAXF.NS", "JSL.NS", "SUZLON.NS", "APLAPOLLO.NS", "PHOENIXLTD.NS", "BIOCON.NS", "UPL.NS", "SCHAEFFLER.NS", "LINDEINDIA.NS", "SUNDARMFIN.NS", "NAM-INDIA.NS", "OFSS.NS", "COLPAL.NS", "ATGL.NS", "PATANJALI.NS", "GLENMARK.NS", "OBEROIRLTY.NS", "ABBOTINDIA.NS", "GODREJPROP.NS", "JSWINFRA.NS", "BERGEPAINT.NS", "FACT.NS", "M&MFIN.NS", "BALKRISIND.NS", "VOLTAS.NS", "MAHABANK.NS", "IRCTC.NS", "TIINDIA.NS", "SUPREMEIND.NS", "PIIND.NS", "TATACOMM.NS", "MOTILALOFS.NS", "MPHASIS.NS", "COFORGE.NS", "360ONE.NS", "BDL.NS", "GSK.NS", "KEI.NS", "PETRONET.NS", "JKCEMENT.NS", "KALYANKJIL.NS", "UBL.NS"]
process_nifty_segment(midcap_pt2, "Midcap150_Part2.xlsx")
process_nifty_car_final(midcap_pt2, "Midcap150_Part2_CAR.xlsx")

Processing: SAIL.NS...
Processing: YESBANK.NS...
Processing: PRESTIGE.NS...
Processing: LLOYDSME.NS...
Processing: ALKEM.NS...
Processing: RVNL.NS...
Processing: NATIONALUM.NS...
Processing: MRF.NS...
Processing: MAXF.NS...


$MAXF.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Processing: JSL.NS...
Processing: SUZLON.NS...
Processing: APLAPOLLO.NS...
Processing: PHOENIXLTD.NS...
Processing: BIOCON.NS...
Processing: UPL.NS...
Processing: SCHAEFFLER.NS...
Processing: LINDEINDIA.NS...
Processing: SUNDARMFIN.NS...
Processing: NAM-INDIA.NS...
Processing: OFSS.NS...
Processing: COLPAL.NS...
Processing: ATGL.NS...
Processing: PATANJALI.NS...
Processing: GLENMARK.NS...
Processing: OBEROIRLTY.NS...
Processing: ABBOTINDIA.NS...
Processing: GODREJPROP.NS...
Processing: JSWINFRA.NS...
Processing: BERGEPAINT.NS...
Processing: FACT.NS...
Processing: M&MFIN.NS...
Processing: BALKRISIND.NS...
Processing: VOLTAS.NS...
Processing: MAHABANK.NS...
Processing: IRCTC.NS...
Processing: TIINDIA.NS...
Processing: SUPREMEIND.NS...
Processing: PIIND.NS...
Processing: TATACOMM.NS...
Processing: MOTILALOFS.NS...
Processing: MPHASIS.NS...
Processing: COFORGE.NS...
Processing: 360ONE.NS...
Processing: BDL.NS...
Processing: GSK.NS...


$GSK.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Processing: KEI.NS...
Processing: PETRONET.NS...
Processing: JKCEMENT.NS...
Processing: KALYANKJIL.NS...
Processing: UBL.NS...
Success! Midcap150_Part2.xlsx created.
Analyzing SAIL.NS...
Analyzing YESBANK.NS...
Analyzing PRESTIGE.NS...
Analyzing LLOYDSME.NS...
Analyzing ALKEM.NS...
Analyzing RVNL.NS...
Analyzing NATIONALUM.NS...
Analyzing MRF.NS...
Analyzing MAXF.NS...


$MAXF.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Analyzing JSL.NS...
Analyzing SUZLON.NS...
Analyzing APLAPOLLO.NS...
Analyzing PHOENIXLTD.NS...
Analyzing BIOCON.NS...
Analyzing UPL.NS...
Analyzing SCHAEFFLER.NS...
Analyzing LINDEINDIA.NS...
Analyzing SUNDARMFIN.NS...
Analyzing NAM-INDIA.NS...
Analyzing OFSS.NS...
Analyzing COLPAL.NS...
Analyzing ATGL.NS...
Analyzing PATANJALI.NS...
Analyzing GLENMARK.NS...
Analyzing OBEROIRLTY.NS...
Analyzing ABBOTINDIA.NS...
Analyzing GODREJPROP.NS...
Analyzing JSWINFRA.NS...
Analyzing BERGEPAINT.NS...
Analyzing FACT.NS...
Analyzing M&MFIN.NS...
Analyzing BALKRISIND.NS...
Analyzing VOLTAS.NS...
Analyzing MAHABANK.NS...
Analyzing IRCTC.NS...
Analyzing TIINDIA.NS...
Analyzing SUPREMEIND.NS...
Analyzing PIIND.NS...
Analyzing TATACOMM.NS...
Analyzing MOTILALOFS.NS...
Analyzing MPHASIS.NS...
Analyzing COFORGE.NS...
Analyzing 360ONE.NS...
Analyzing BDL.NS...
Analyzing GSK.NS...


$GSK.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Analyzing KEI.NS...
Analyzing PETRONET.NS...
Analyzing JKCEMENT.NS...
Analyzing KALYANKJIL.NS...
Analyzing UBL.NS...
Success! Midcap150_Part2_CAR.xlsx created.


In [58]:
midcap_pt3 = ["ASTRAL.NS", "BLUESTARCO.NS", "DALBHARAT.NS", "ESCORTS.NS", "3MINDIA.NS", "HUDCO.NS", "COCHINSHIP.NS", "APARINDS.NS", "PGHH.NS", "CONCOR.NS", "IPCALAB.NS", "PAGEIND.NS", "GUJFLUORO.NS", "LTTS.NS", "AJANTPHARM.NS", "AIAENG.NS", "NLCINDIA.NS", "UCOBANK.NS", "JUBLFOOD.NS", "ENDURANCE.NS", "IREDA.NS", "PREMIER.NS", "THERMAX.NS", "SONACOMS.NS", "GODREJIND.NS", "CRISIL.NS", "GODFRYPHLP.NS", "TATAINVEST.NS", "KPRMILL.NS", "APOLLOTYRE.NS", "ACC.NS", "MEDANTA.NS", "SJVN.NS", "TATAELXSI.NS", "GUJGASLTD.NS", "EXIDEIND.NS", "LICHSGFIN.NS", "HONAUT.NS", "IRB.NS", "AWL.NS", "NIACL.NS", "TATATECH.NS", "KPITTECH.NS", "IGL.NS", "DEEPAKNTR.NS", "SYNGENE.NS"]
process_nifty_segment(midcap_pt3, "Midcap150_Part3.xlsx")
process_nifty_car_final(midcap_pt3, "Midcap150_Part3_CAR.xlsx")

Processing: ASTRAL.NS...
Processing: BLUESTARCO.NS...
Processing: DALBHARAT.NS...
Processing: ESCORTS.NS...
Processing: 3MINDIA.NS...
Processing: HUDCO.NS...
Processing: COCHINSHIP.NS...
Processing: APARINDS.NS...
Processing: PGHH.NS...
Processing: CONCOR.NS...
Processing: IPCALAB.NS...
Processing: PAGEIND.NS...
Processing: GUJFLUORO.NS...


$GUJFLUORO.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Processing: LTTS.NS...
Processing: AJANTPHARM.NS...
Processing: AIAENG.NS...
Processing: NLCINDIA.NS...
Processing: UCOBANK.NS...
Processing: JUBLFOOD.NS...
Processing: ENDURANCE.NS...
Processing: IREDA.NS...
Processing: PREMIER.NS...
Processing: THERMAX.NS...
Processing: SONACOMS.NS...
Processing: GODREJIND.NS...
Processing: CRISIL.NS...
Processing: GODFRYPHLP.NS...
Processing: TATAINVEST.NS...
Processing: KPRMILL.NS...
Processing: APOLLOTYRE.NS...
Processing: ACC.NS...
Processing: MEDANTA.NS...
Processing: SJVN.NS...
Processing: TATAELXSI.NS...
Processing: GUJGASLTD.NS...
Processing: EXIDEIND.NS...
Processing: LICHSGFIN.NS...
Processing: HONAUT.NS...
Processing: IRB.NS...
Processing: AWL.NS...
Processing: NIACL.NS...
Processing: TATATECH.NS...
Processing: KPITTECH.NS...
Processing: IGL.NS...
Processing: DEEPAKNTR.NS...
Processing: SYNGENE.NS...
Success! Midcap150_Part3.xlsx created.
Analyzing ASTRAL.NS...
Analyzing BLUESTARCO.NS...
Analyzing DALBHARAT.NS...
Analyzing ESCORTS.NS...
An

$GUJFLUORO.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Analyzing LTTS.NS...
Analyzing AJANTPHARM.NS...
Analyzing AIAENG.NS...
Analyzing NLCINDIA.NS...
Analyzing UCOBANK.NS...
Analyzing JUBLFOOD.NS...
Analyzing ENDURANCE.NS...
Analyzing IREDA.NS...
Analyzing PREMIER.NS...
Analyzing THERMAX.NS...
Analyzing SONACOMS.NS...
Analyzing GODREJIND.NS...
Analyzing CRISIL.NS...
Analyzing GODFRYPHLP.NS...
Analyzing TATAINVEST.NS...
Analyzing KPRMILL.NS...
Analyzing APOLLOTYRE.NS...
Analyzing ACC.NS...
Analyzing MEDANTA.NS...
Analyzing SJVN.NS...
Analyzing TATAELXSI.NS...
Analyzing GUJGASLTD.NS...
Analyzing EXIDEIND.NS...
Analyzing LICHSGFIN.NS...
Analyzing HONAUT.NS...
Analyzing IRB.NS...
Analyzing AWL.NS...
Analyzing NIACL.NS...
Analyzing TATATECH.NS...
Analyzing KPITTECH.NS...
Analyzing IGL.NS...
Analyzing DEEPAKNTR.NS...
Analyzing SYNGENE.NS...
Success! Midcap150_Part3_CAR.xlsx created.


In [59]:
# Combined list from your image and previous request
portfolio_list = [
    # --- From Image ---
    'ITC.NS', 'TCS.NS', 'CHAMBLFERT.NS', 'HAL.NS', 'BHARATCOAL.NS', 
    'TATATECH.NS', 'NMDC.NS', 'SJVN.NS', 'JIOFIN.NS', 'VBL.NS', 
    'WIPRO.NS', 'NLCINDIA.NS', 'HINDCOPPER.NS', 'DEVYANI.NS', 'RECLTD.NS',
    'NHPC.NS', 'DELTACORP.NS', 'OLAELEC.NS', 'DHAMPURSUG.NS', 'IOC.NS', 
    'ADANIGREEN.NS', 'HAPPSTMNDS.NS', 'BEL.NS', 'ICICIPRULI.NS', 
    'EASEMYTRIP.NS', 'OLECTRA.NS', 'JSWINFRA.NS', 'AVALON.NS', 
    'MARICO.NS', 'WAAREEENER.NS',
    
    # --- From Previous Midcap List ---
    'ANGELONE.NS', 'ASIANPAINT.NS', 'DIXON.NS', 'EDELWEISS.NS', 
    'EQUITASBNK.NS', 'GOLDIETF-E.NS', 'HDFCBANK.NS', 'HEROMOTOCO.NS', 
    'IRCTC.NS', 'IRFC.NS', 'KALPATARU.NS', 'NORTHARC.NS', 'NTPC.NS', 
    'ONGC.NS', 'RVNL.NS', 'TATAPOWER.NS', 'TITAGARH.NS', 'TITAN.NS', 
    'TMCV.NS', 'TMPV.NS'
]

# Run the final CAR strategy function
process_nifty_segment(portfolio_list, "Portfolio_list.xlsx")
process_nifty_car_final(portfolio_list, "Comprehensive_CAR_Report.xlsx")

Processing: ITC.NS...
Processing: TCS.NS...
Processing: CHAMBLFERT.NS...
Processing: HAL.NS...
Processing: BHARATCOAL.NS...
Processing: TATATECH.NS...
Processing: NMDC.NS...
Processing: SJVN.NS...
Processing: JIOFIN.NS...
Processing: VBL.NS...
Processing: WIPRO.NS...
Processing: NLCINDIA.NS...
Processing: HINDCOPPER.NS...
Processing: DEVYANI.NS...
Processing: RECLTD.NS...
Processing: NHPC.NS...
Processing: DELTACORP.NS...
Processing: OLAELEC.NS...
Processing: DHAMPURSUG.NS...
Processing: IOC.NS...
Processing: ADANIGREEN.NS...
Processing: HAPPSTMNDS.NS...
Processing: BEL.NS...
Processing: ICICIPRULI.NS...
Processing: EASEMYTRIP.NS...
Processing: OLECTRA.NS...
Processing: JSWINFRA.NS...
Processing: AVALON.NS...
Processing: MARICO.NS...
Processing: WAAREEENER.NS...
Processing: ANGELONE.NS...
Processing: ASIANPAINT.NS...
Processing: DIXON.NS...
Processing: EDELWEISS.NS...
Processing: EQUITASBNK.NS...
Processing: GOLDIETF-E.NS...


$GOLDIETF-E.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Processing: HDFCBANK.NS...
Processing: HEROMOTOCO.NS...
Processing: IRCTC.NS...
Processing: IRFC.NS...
Processing: KALPATARU.NS...
Processing: NORTHARC.NS...
Processing: NTPC.NS...
Processing: ONGC.NS...
Processing: RVNL.NS...
Processing: TATAPOWER.NS...
Processing: TITAGARH.NS...
Processing: TITAN.NS...
Processing: TMCV.NS...
Processing: TMPV.NS...
Success! Portfolio_list.xlsx created.
Analyzing ITC.NS...
Analyzing TCS.NS...
Analyzing CHAMBLFERT.NS...
Analyzing HAL.NS...
Analyzing BHARATCOAL.NS...
Analyzing TATATECH.NS...
Analyzing NMDC.NS...
Analyzing SJVN.NS...
Analyzing JIOFIN.NS...
Analyzing VBL.NS...
Analyzing WIPRO.NS...
Analyzing NLCINDIA.NS...
Analyzing HINDCOPPER.NS...
Analyzing DEVYANI.NS...
Analyzing RECLTD.NS...
Analyzing NHPC.NS...
Analyzing DELTACORP.NS...
Analyzing OLAELEC.NS...
Analyzing DHAMPURSUG.NS...
Analyzing IOC.NS...
Analyzing ADANIGREEN.NS...
Analyzing HAPPSTMNDS.NS...
Analyzing BEL.NS...
Analyzing ICICIPRULI.NS...
Analyzing EASEMYTRIP.NS...
Analyzing OLECTRA.N

$GOLDIETF-E.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


Analyzing HDFCBANK.NS...
Analyzing HEROMOTOCO.NS...
Analyzing IRCTC.NS...
Analyzing IRFC.NS...
Analyzing KALPATARU.NS...
Analyzing NORTHARC.NS...
Analyzing NTPC.NS...
Analyzing ONGC.NS...
Analyzing RVNL.NS...
Analyzing TATAPOWER.NS...
Analyzing TITAGARH.NS...
Analyzing TITAN.NS...
Analyzing TMCV.NS...
Analyzing TMPV.NS...
Success! Comprehensive_CAR_Report.xlsx created.
